In [15]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import zscore
import numpy as np

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error as RMSE

from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

In [6]:

train_data = pd.read_csv('/Users/iris666/Projects/Kaggle/Predict_Podcast_Listening_Time/train.csv')

test_data = pd.read_csv('/Users/iris666/Projects/Kaggle/Predict_Podcast_Listening_Time/test.csv')

In [7]:
### Data Cleaning
## Drop the 'id' column as it is not needed for analysis
train_data.drop(['id'], axis=1, inplace=True)
test_data.drop(['id'], axis=1, inplace=True)
## Drop records (rows) with any null values
train_data.dropna(inplace=True)

In [8]:
# Identify outlier rows using Z-score > 3 in any numeric column and drop them
z_scores = np.abs(zscore(train_data.select_dtypes(include='number')))
outlier_mask = (z_scores > 3).any(axis=1)
train_data = train_data[~outlier_mask]

In [9]:
## Prepare training data features and target variable
X = train_data.drop('Listening_Time_minutes', axis=1) #all features except target
y = train_data['Listening_Time_minutes'] #target variable

In [10]:
## One-Hot Encode categorical variables in the training data
ohe = OneHotEncoder(handle_unknown='ignore', ## ignore null
                    sparse_output=False).set_output(transform="pandas")
ohetransform = ohe.fit_transform(X.select_dtypes(exclude='number'))

In [11]:
proc_X = pd.concat([ohetransform, X.select_dtypes(include='number')], axis=1)
X = proc_X

In [12]:
## Split dataset into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

XGBoost Model Tuning Run 1

In [13]:

# Reduce training data size for faster tuning if needed
X_train_xgb = X_train.sample(n=50000, random_state=42) if len(X_train) > 50000 else X_train
y_train_xgb = y_train.loc[X_train_xgb.index] if len(X_train) > 50000 else y_train

# Define parameter grid for XGBoost
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [6, 10, 15],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1, tree_method='hist')
grid = GridSearchCV(xgb, param_grid, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=1)
grid.fit(X_train_xgb, y_train_xgb)

print('Best parameters:', grid.best_params_)
print('Best RMSE:', -grid.best_score_)

Fitting 3 folds for each of 72 candidates, totalling 216 fits


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 200, 'subsample': 0.8}
Best RMSE: 10.588082867212991


In [14]:
proc_test_X = pd.concat([ohe.transform(test_data.select_dtypes(exclude='number')), 
                         test_data.select_dtypes(include='number')], axis=1)
## Handle missing values in test dataset using SimpleImputer
imputer = SimpleImputer(strategy='mean')
test_X_imputed = imputer.fit_transform(proc_test_X)

test_predictions = grid.predict(test_X_imputed)
submission = pd.DataFrame({
    'id': test_data.index + 750000,  # Adjusting index to match original IDs
    'Listening_Time_minutes': test_predictions
})
submission.to_csv('submission3.csv', index=False, header = True)

XGBoost Model Tuning Run 2

In [17]:
param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 6, 10, 15],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2, 0.5],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [1, 1.5, 2, 3]
}

xgb = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=-1, tree_method='hist')
random_search = RandomizedSearchCV(
    xgb, param_distributions=param_dist, n_iter=30, 
    scoring='neg_root_mean_squared_error', cv=5, n_jobs=-1, verbose=1, random_state=42
)
random_search.fit(X_train_xgb, y_train_xgb)
print('Best parameters:', random_search.best_params_)
print('Best RMSE:', -random_search.best_score_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05, 'gamma': 0.5, 'colsample_bytree': 0.8}
Best RMSE: 10.57158425660916
Best parameters: {'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 200, 'max_depth': 6, 'learning_rate': 0.05, 'gamma': 0.5, 'colsample_bytree': 0.8}
Best RMSE: 10.57158425660916


In [18]:
proc_test_X = pd.concat([ohe.transform(test_data.select_dtypes(exclude='number')), 
                         test_data.select_dtypes(include='number')], axis=1)
## Handle missing values in test dataset using SimpleImputer
imputer = SimpleImputer(strategy='mean')
test_X_imputed = imputer.fit_transform(proc_test_X)

test_predictions = grid.predict(test_X_imputed)
submission = pd.DataFrame({
    'id': test_data.index + 750000,  # Adjusting index to match original IDs
    'Listening_Time_minutes': test_predictions
})
submission.to_csv('submission4.csv', index=False, header = True)